# 01 — Coleta, Rotulagem e Splits**TCC: Detecção de Smishing em Idosos com Modelos de Linguagem Natural**Constrói a base unificada e produz o **split único 70/15/15 (seed 42)** quetodos os modelos compartilham. A integridade desse split é o que sustenta aafirmação de que as trilhas foram comparadas em condições iguais.### A decisão mais importante do projeto**Split primeiro, sintéticas depois — e só no treino.**```corpus real → split 70/15/15 estratificado                    │                    ├── train ← recebe as mensagens sintéticas                    ├── val   ← 100% real                    └── test  ← 100% real```Assim a monografia pode afirmar que *os modelos foram avaliados exclusivamentesobre mensagens reais*. Isso responde de uma vez a duas perguntas de banca: a dovazamento de dados e a de "seu modelo não está apenas reconhecendo o estilo dogerador?".Fazer o contrário — aumentar e depois dividir — coloca variações da mesmamensagem-semente em treino e teste. Todas as métricas sobem, os três modelosparecem ótimos, e o resultado não vale nada.### Saídas| Arquivo | Conteúdo ||---|---|| `data/processed/corpus_completo.csv` | `id, texto, rotulo, tipo_golpe, id_semente, fonte, idioma` || `data/splits/train.csv`, `val.csv`, `test.csv` | Split fixo, com todas as colunas acima || `data/splits/split_info.json` | Seed, proporções, contagens, nº de sintéticas |

## 1. Setup

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
# No Colab CADA notebook roda em um runtime próprio: instalar dependências
# em um notebook não vale para os outros. Por isso esta célula se repete em
# todos, e não existe um "notebook de instalação".

REPO = 'https://github.com/SEU-USUARIO/tcc-smishing.git'   # ← ajuste aqui

!git clone -q {REPO} /content/tcc-smishing 2>/dev/null || (cd /content/tcc-smishing && git pull -q)
!pip install -q -r /content/tcc-smishing/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/tcc-smishing/src')

import config as CFG
CFG.fixar_seeds()
CFG.criar_pastas()
CFG.resumo()

In [ ]:
import os
import pandas as pd
import numpy as np

import preprocessing as pp
import evaluation as ev

RAW = CFG.PATHS['raw']


def padronizar(df, fonte, idioma, tipo_padrao='outro'):
    """Converte qualquer fonte para o esquema único do projeto."""
    df = df.copy()
    df = df.dropna(subset=[CFG.COL_TEXTO])
    df = df[df[CFG.COL_TEXTO].astype(str).str.strip() != '']

    if 'tipo_golpe' not in df.columns:
        df['tipo_golpe'] = np.where(
            df[CFG.COL_ROTULO] == CFG.CLASSE_POSITIVA, tipo_padrao, 'nao_aplicavel')

    if 'id_semente' not in df.columns:
        df['id_semente'] = ''      # mensagens reais não têm semente

    df = df.reset_index(drop=True)
    df['id'] = [f'{fonte}_{i}' for i in range(len(df))]
    df['fonte'] = fonte
    df['idioma'] = idioma

    return df[['id', CFG.COL_TEXTO, CFG.COL_ROTULO, 'tipo_golpe',
               'id_semente', 'fonte', 'idioma']]


def mapear_rotulos(serie):
    """Aplica o mapa do config, devolvendo NaN para rótulos desconhecidos."""
    return serie.astype(str).str.strip().str.lower().map(CFG.MAPA_ROTULOS)


print('Utilitários definidos.')

## 2. SMS Spam Collection (UCI)> **Papel: validação de pipeline apenas.** É spam genérico em inglês, não> smishing em português. Serve para confirmar que a esteira de pré-processamento> e modelagem funciona mecanicamente. **Não entra no corpus principal e não> constitui achado do domínio do TCC** — no texto, esses resultados precisam> aparecer claramente separados.

In [ ]:
from ucimlrepo import fetch_ucirepo

sms = fetch_ucirepo(id=228)
df_sms = pd.DataFrame({
    CFG.COL_TEXTO: sms.data.features['sms'],
    # 'spam' aqui é mensagem indesejada; mapeada para a classe positiva apenas
    # para que o pipeline mecânico tenha duas classes com que trabalhar
    CFG.COL_ROTULO: sms.data.targets['label'].map(
        {'spam': CFG.CLASSE_POSITIVA, 'ham': CFG.CLASSE_NEGATIVA}),
})

df_sms = padronizar(df_sms, fonte='sms_spam_collection', idioma='en')
df_sms.to_csv(f'{RAW}/sms_spam_collection.csv', index=False, encoding='utf-8')

print(f'SMS Spam Collection: {len(df_sms)} mensagens')
print(df_sms[CFG.COL_ROTULO].value_counts().to_string())

## 3. Fontes curadas em PT-BRBortot et al. (2024) e CERT.br / GOV.BR **não são datasets prontos** — sãomateriais curados que compõem o corpus principal. Faça o upload dos CSVs em`data/raw/` antes de executar.Colunas esperadas: `texto` (ou `mensagem`), `rotulo` e, idealmente, `tipo_golpe`.

In [ ]:
# Ajuste os nomes de coluna conforme os arquivos reais
FONTES_PT = [
    {'arquivo': 'bortot.csv', 'fonte': 'bortot', 'col_texto': 'mensagem', 'col_rotulo': 'rotulo'},
    {'arquivo': 'certbr.csv', 'fonte': 'certbr', 'col_texto': 'mensagem', 'col_rotulo': 'rotulo'},
]


def carregar_fonte(spec):
    caminho = f"{RAW}/{spec['arquivo']}"
    if not os.path.isfile(caminho):
        print(f"[AVISO] {spec['arquivo']} não encontrado em data/raw/ — fonte ignorada.")
        return None

    bruto = pd.read_csv(caminho, encoding='utf-8')
    print(f"{spec['arquivo']}: colunas {bruto.columns.tolist()}")

    df = pd.DataFrame({
        CFG.COL_TEXTO:  bruto[spec['col_texto']],
        CFG.COL_ROTULO: mapear_rotulos(bruto[spec['col_rotulo']]),
    })
    if 'tipo_golpe' in bruto.columns:
        df['tipo_golpe'] = bruto['tipo_golpe']

    antes = len(df)
    df = df.dropna(subset=[CFG.COL_ROTULO])
    if len(df) < antes:
        print(f'  [INFO] {antes - len(df)} linhas descartadas por rótulo desconhecido')

    return padronizar(df, fonte=spec['fonte'], idioma='pt')


fontes_reais = [df for df in (carregar_fonte(s) for s in FONTES_PT) if df is not None]

if not fontes_reais:
    raise FileNotFoundError(
        'Nenhuma fonte PT-BR disponível. Faça o upload dos CSVs curados em data/raw/.'
    )

reais = pd.concat(fontes_reais, ignore_index=True)
print(f'\nCorpus real PT-BR: {len(reais)} mensagens')
print(reais[CFG.COL_ROTULO].value_counts().to_string())

## 4. DeduplicaçãoDeduplicar por texto exato deixa passar variações triviais de espaçamento,pontuação e caixa. Como as fontes se sobrepõem, a comparação é feita sobre aforma normalizada.

In [ ]:
reais['_chave'] = reais[CFG.COL_TEXTO].apply(pp.chave_dedup)

antes = len(reais)
reais = reais.drop_duplicates(subset='_chave').drop(columns='_chave').reset_index(drop=True)
print(f'Duplicatas removidas: {antes - len(reais)}')
print(f'Corpus real após deduplicação: {len(reais)}')

# IDs definitivos, atribuídos depois da deduplicação
reais['id'] = [f'real_{i}' for i in range(len(reais))]

## 5. Split estratificado — apenas mensagens reaisA estratificação preserva a proporção de classes nos três subconjuntos, o que éessencial dado o desbalanceamento típico de corpora de spam/smishing.

In [ ]:
from sklearn.model_selection import train_test_split

R = CFG.SPLIT_RATIO

temp, teste = train_test_split(
    reais, test_size=R['test'], random_state=CFG.SEED, stratify=reais[CFG.COL_ROTULO],
)
# proporção da validação relativa ao que sobrou: 0.15 / 0.85
treino, val = train_test_split(
    temp, test_size=R['val'] / (R['train'] + R['val']),
    random_state=CFG.SEED, stratify=temp[CFG.COL_ROTULO],
)

treino, val, teste = (d.reset_index(drop=True) for d in (treino, val, teste))

print('=== Split das mensagens reais ===')
for nome, d in [('treino', treino), ('val', val), ('teste', teste)]:
    print(f'  {nome:8}: {len(d):5}  ({len(d)/len(reais):.1%})  '
          f'{(d[CFG.COL_ROTULO] == CFG.CLASSE_POSITIVA).mean():.1%} smishing')

## 6. Incorporação das mensagens sintéticas — só no treinoSe o notebook 00 não foi executado, esta célula é ignorada e o corpus ficaapenas com mensagens reais.

In [ ]:
if os.path.isfile(CFG.SINTETICAS):
    sinteticas = pd.read_csv(CFG.SINTETICAS, encoding='utf-8')
    sinteticas = padronizar(sinteticas, fonte='sintetica', idioma='pt')

    # Filtro 1 — uma variação só entra se a semente dela estiver no TREINO.
    # Se a semente caiu em val ou teste, a variação vaza informação do
    # conjunto de avaliação.
    sementes_treino = set(treino['id']) | set(treino['id_semente'])
    antes = len(sinteticas)
    sinteticas = sinteticas[
        sinteticas['id_semente'].isin(sementes_treino) | (sinteticas['id_semente'] == '')
    ]
    print(f'[INFO] {antes - len(sinteticas)} descartadas — semente fora do treino')

    # Filtro 2 — o modelo gerador pode ter produzido, por acaso, uma mensagem
    # quase idêntica a uma de val ou teste, mesmo partindo de semente do
    # treino. A comparação é sobre a forma normalizada, não sobre o texto
    # exato: variações de espaçamento e pontuação não podem servir de disfarce.
    chaves_avaliacao = (set(val[CFG.COL_TEXTO].apply(pp.chave_dedup)) |
                        set(teste[CFG.COL_TEXTO].apply(pp.chave_dedup)))
    antes = len(sinteticas)
    sinteticas = sinteticas[~sinteticas[CFG.COL_TEXTO].apply(pp.chave_dedup).isin(chaves_avaliacao)]
    print(f'[INFO] {antes - len(sinteticas)} descartadas — texto colide com val/teste')

    # Filtro 3 — redundância com o próprio treino: não agrega, só infla
    chaves_treino = set(treino[CFG.COL_TEXTO].apply(pp.chave_dedup))
    antes = len(sinteticas)
    sinteticas = sinteticas[~sinteticas[CFG.COL_TEXTO].apply(pp.chave_dedup).isin(chaves_treino)]
    print(f'[INFO] {antes - len(sinteticas)} descartadas — já presentes no treino')

    treino = pd.concat([treino, sinteticas], ignore_index=True)
    treino = treino.sample(frac=1, random_state=CFG.SEED).reset_index(drop=True)

    print(f'Sintéticas incorporadas ao treino: {len(sinteticas)}')
else:
    print('[INFO] Sem mensagens sintéticas — corpus apenas com mensagens reais.')

print(f'\nTreino final: {len(treino)}  |  Val: {len(val)}  |  Teste: {len(teste)}')

## 7. Verificações de integridadeQualquer falha aqui invalida todos os resultados a jusante. O notebook para.

In [ ]:
falhas = []

# 7.1 — nenhum id em dois subconjuntos
ids = {'treino': set(treino['id']), 'val': set(val['id']), 'teste': set(teste['id'])}
for a, b in [('treino', 'val'), ('treino', 'teste'), ('val', 'teste')]:
    comum = ids[a] & ids[b]
    print(f'  [{"ok" if not comum else "ERRO"}] {a} ∩ {b}: {len(comum)} ids')
    if comum:
        falhas.append(f'{len(comum)} ids compartilhados entre {a} e {b}')

# 7.2 — nenhuma semente atravessando o split (o vazamento perigoso)
sem = {k: set(d.loc[d['id_semente'].astype(str) != '', 'id_semente'])
       for k, d in [('treino', treino), ('val', val), ('teste', teste)]}
for a, b in [('treino', 'val'), ('treino', 'teste'), ('val', 'teste')]:
    comum = sem[a] & sem[b]
    print(f'  [{"ok" if not comum else "ERRO"}] sementes {a} ∩ {b}: {len(comum)}')
    if comum:
        falhas.append(f'{len(comum)} sementes compartilhadas entre {a} e {b}')

# 7.3 — val e teste precisam ser 100% reais
for nome, d in [('val', val), ('teste', teste)]:
    n_sint = int((d['fonte'] == 'sintetica').sum())
    print(f'  [{"ok" if not n_sint else "ERRO"}] {nome} sem sintéticas: {n_sint} encontradas')
    if n_sint:
        falhas.append(f'{n_sint} sintéticas em {nome}')

# 7.4 — texto duplicado atravessando o split
chaves = {k: set(d[CFG.COL_TEXTO].apply(pp.chave_dedup))
          for k, d in [('treino', treino), ('val', val), ('teste', teste)]}
for a, b in [('treino', 'val'), ('treino', 'teste'), ('val', 'teste')]:
    comum = chaves[a] & chaves[b]
    print(f'  [{"ok" if not comum else "ERRO"}] textos {a} ∩ {b}: {len(comum)}')
    if comum:
        falhas.append(f'{len(comum)} textos idênticos entre {a} e {b}')

if falhas:
    raise AssertionError('Split inválido:\n  - ' + '\n  - '.join(falhas))
print('\n[OK] Split íntegro.')

## 8. Salvamento

In [ ]:
corpus = pd.concat([treino, val, teste], ignore_index=True)
corpus.to_csv(CFG.CORPUS_COMPLETO, index=False, encoding='utf-8')
print(f'Corpus completo: {CFG.CORPUS_COMPLETO}  ({len(corpus)} mensagens)')

splits = {'train': treino, 'val': val, 'test': teste}
for nome, d in splits.items():
    d.to_csv(CFG.SPLIT_FILES[nome], index=False, encoding='utf-8')
    print(f'  {nome}.csv  ({len(d)} linhas)')

ev.salvar_split_info(splits, extras={
    'fontes_reais': [s['fonte'] for s in FONTES_PT],
    'sinteticas_no_treino': int((treino['fonte'] == 'sintetica').sum()),
    'val_teste_apenas_reais': True,
})

## 9. Estatísticas e figuras

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribuição de classes por subconjunto
for ax, (nome, d) in zip(axes, splits.items()):
    contagem = d[CFG.COL_ROTULO].value_counts()
    barras = ax.bar(contagem.index, contagem.values, color=['#d62728', '#2ca02c'])
    ax.set_title(f'{nome} ({len(d)} msgs)')
    ax.set_ylabel('Quantidade')
    for barra in barras:
        ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height(),
                str(int(barra.get_height())), ha='center', va='bottom', fontweight='bold')

plt.suptitle(f'Distribuição de classes por subconjunto (seed={CFG.SEED})')
plt.tight_layout()
plt.savefig(f"{CFG.PATHS['figures']}/01_split_classes.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tipos de golpe e comprimento das mensagens — insumo da etapa 4.4.1
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

golpes = corpus[corpus[CFG.COL_ROTULO] == CFG.CLASSE_POSITIVA]['tipo_golpe'].value_counts()
axes[0].barh(golpes.index, golpes.values, color='#4878CF')
axes[0].set_title('Mensagens por tipo de golpe')
axes[0].set_xlabel('Quantidade')

corpus['n_chars'] = corpus[CFG.COL_TEXTO].str.len()
for rotulo, cor in [(CFG.CLASSE_POSITIVA, '#d62728'), (CFG.CLASSE_NEGATIVA, '#2ca02c')]:
    axes[1].hist(corpus.loc[corpus[CFG.COL_ROTULO] == rotulo, 'n_chars'],
                 bins=40, alpha=0.6, label=rotulo, color=cor)
axes[1].set_title('Comprimento das mensagens por classe')
axes[1].set_xlabel('Número de caracteres')
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{CFG.PATHS['figures']}/01_corpus_descritivo.png", dpi=150, bbox_inches='tight')
plt.show()

print(corpus.groupby(CFG.COL_ROTULO)['n_chars'].describe().round(1).to_string())
print('\nProssiga para o notebook 02_baseline_classico.ipynb.')